## Homework: Vector Search

In this homework, we put what we learned in Module 2 into practice.

We'll first turn text into vectors, then search by similarity.
We'll also learn something new and see how to combine vector search with keyword search. We'll skip the RAG part and focus solely on search.

## Setup

In this homework we won't use the same approach for embedding as in the
module. That is, we won't use the sentence-transformers library. Instead,
we'll use the lightweight embedding approach with the ONNX `Embedder`.

Both approaches produce identical vectors, but the ONNX runtime is far
lighter. It needs no PyTorch and no CUDA, which makes the installation about
30x smaller and lets it run anywhere, including a basic Codespace. We
skimmed through it in the lesson and said we'd cover it in the homework -
so here we are.

```bash
uv add onnxruntime tokenizers
```

By default `download.py` fetches `Xenova/all-MiniLM-L6-v2`, the ONNX
version of the `all-MiniLM-L6-v2` model from the lessons:

In [2]:
!uv run python download.py

tokenizer.json: 100%|████████████████████████| 712k/712k [00:00<00:00, 97.4MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json

onnx/model.onnx: downloading bytes: █████████████████████▌  | 81.1MB, 6.92MB/s  
onnx/model.onnx: downloading bytes: ████████████████████████| 84.7MB, 7.87MB/s  
onnx/model.onnx: reconstructing file: 100%|████████| 90.4MB / 90.4MB, 8.70MB/s  
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

In [3]:
from embedder import Embedder

embed = Embedder()

2026-08-19 09:46:46.275160798 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [6]:
q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)
v1[0]

np.float64(-0.02058203437252893)

## Loading the data

We pull the lesson pages from the course repository, the same way as in
homework 1. We pin to commit `8c1834d` so everyone works with the same
data.

In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [9]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Сompute the cosine similarity with the query vector
from Q1. What do you get?

In [16]:
q = [doc for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md']
content = q[0]['content']

v_content = embed.encode(content)

In [17]:
v1.dot(v_content)

np.float64(0.36107026789538205)

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

In [24]:
import numpy as np

In [18]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's `content` with `encode_batch`, stack the vectors
into a matrix `X`, and score the Q1 query against all chunks:

In [21]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [28]:
len(chunks)

295

In [40]:
X = np.array([embed.encode(chunk['content']) for chunk in chunks])
X.shape

(295, 384)

In [29]:
scores = X.dot(v1)

Which file does the highest-scoring chunk belong to (its filename)

In [34]:
chunks[np.argmax(scores)]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:


In [35]:
from minsearch import VectorSearch

In [42]:
v_index = VectorSearch(keyword_fields=['filename'])
v_index.fit(X, chunks)

In [45]:
q2 = 'What metric do we use to evaluate a search engine?'
v2 = embed.encode(q2)

In [46]:
v_index.search(v2, num_results=1)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

In [52]:
from minsearch import Index

In [53]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

In [55]:
q3 = 'How do I store vectors in PostgreSQL?'
v3 = embed.encode(q3)

In [56]:
text_search_results = index.search(
    q3,
    num_results=5
)

text_search_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

In [58]:
vector_search_results = v_index.search(v3, num_results=5)

vector_search_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [59]:
def get_filenames(results):
    return set([result['filename'] for result in results])   

In [62]:
get_filenames(vector_search_results) - get_filenames(text_search_results)

{'02-vector-search/lessons/08-pgvector.md'}

## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

In [66]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["content"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:

In [67]:
q4 = "How do I give the model access to tools?"
v4 = embed.encode(q4)

In [75]:
vector_results = v_index.search(v4, num_results=10)
text_results = index.search(q4, num_results=10)

In [76]:
results = rrf([text_results, vector_results], num_results=5)
results

[{'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson we built a RAG pipeline with `RAGBase.rag()`\nand saw it fail on the "Olama" typo. The search returned nothing\nuseful, and the LLM had no way to recover.\n\nThe flow that broke:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    S[search - Olama - no useful results]\n    A([LLM: I don\'t have information about Olama.])\n\n    U --> S --> A\n```\n\nThe pipeline is fixed: search, build prompt, LLM.\n\n```python\ndef rag(self, query):\n    search_results = self.search(query)\n    prompt = self.build_prompt(query, search_results)\n    answer = self.llm(prompt)\n    return answer\n```\n\nThe LLM is a passenger here, not a driver. It never sees the bad\nsearch results, so it can\'t try again with a corrected query.\n\n## The agent alternative\n\nAn agent puts the LLM in charge.\n\nInstead of running se